# ECLSS — Life Support Sizing Demo

Compute ECLSS requirements for crewed habitats using NASA BVAD reference data.

**Scenario:** Lunar surface habitat, 4 crew, 25 m³, 180-day mission.

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from eclss import (
    Habitat, CrewMember, CrewMetabolicProfile,
    Simulation, EclssSystem, ContaminantEstimator,
    assess_system, MinimumRequirements,
    ConstantSchedule, ISS_WEEKLY, REFERENCE_SCHEDULE,
    ActivityLevel, DailySchedule, ActivityBlock, SLEEP, NOMINAL,
)

## 1. Crew Profiles

Each crew member gets BVAD-scaled metabolic rates from BMR equations. Override any rate with `CrewMetabolicProfile`.

In [ ]:
crew = [
    CrewMember(name="Kirk",   age=34, mass_kg=82, height_m=1.78, sex="male"),
    CrewMember(name="Uhura",  age=29, mass_kg=61, height_m=1.68, sex="female"),
    CrewMember(name="Spock",  age=35, mass_kg=77, height_m=1.91, sex="male"),
    CrewMember(name="Scotty", age=44, mass_kg=85, height_m=1.75, sex="male",
               profile=CrewMetabolicProfile(potable_water_kg_day=4.0)),  # higher water intake
]

# Compare metabolic profiles
names = [cm.name for cm in crew]
o2 = np.array([cm.daily_o2_kg for cm in crew])
co2 = np.array([cm.daily_co2_kg for cm in crew])
food = np.array([cm.daily_food_kg for cm in crew])
water = np.array([cm.daily_potable_water_kg for cm in crew])

fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(14, 4))
for ax, data, label, unit in zip(axes, [o2, co2, food, water],
    ["O2 Consumed", "CO2 Produced", "Food (dry)", "Potable Water"],
    ["kg/day"] * 4):
    bars = ax.bar(names, data, color=["#2196F3", "#E91E63", "#4CAF50", "#FF9800"])
    ax.set_ylabel(unit)
    ax.set_title(label)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.suptitle("Per-Crew Daily Metabolic Rates (BVAD-scaled)", y=1.02, fontsize=13)
plt.show()

print(f"\nScotty's water override: {crew[3].daily_potable_water_kg:.1f} kg/day (others: {crew[0].daily_potable_water_kg:.2f})")

## 2. Open vs Closed Loop Comparison

In [ ]:
hab = Habitat(crew=crew, cabin_volume_m3=25.0, mission_duration_days=180)
r_open = Habitat(crew=crew, cabin_volume_m3=25.0, mission_duration_days=180, loop_mode="open").calculate()
r_closed = Habitat(crew=crew, cabin_volume_m3=25.0, mission_duration_days=180, loop_mode="closed").calculate()

# Side-by-side comparison
categories = ["O2", "Water", "Food"]
open_vals = np.array([r_open.total_o2_kg, r_open.total_water_kg, r_open.total_food_kg])
closed_vals = np.array([r_closed.total_o2_kg, r_closed.total_water_kg, r_closed.total_food_kg])

x = np.arange(len(categories))
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 5))
ax.bar(x - 0.2, open_vals, width=0.35, label="Open-loop", color="#EF5350")
ax.bar(x + 0.2, closed_vals, width=0.35, label="Closed-loop", color="#66BB6A")
ax.set_ylabel("Mission Total (kg)")
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.set_title("180-Day Mission Consumables: Open vs Closed Loop")

savings = 1 - closed_vals.sum() / open_vals.sum()
ax.text(0.98, 0.95, f"Mass savings: {savings:.0%}", transform=ax.transAxes,
        ha="right", va="top", fontsize=12, fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="#C8E6C9"))
plt.tight_layout()
plt.show()

print(f"\nOpen-loop:   {open_vals.sum():,.0f} kg")
print(f"Closed-loop: {closed_vals.sum():,.0f} kg")

## 3. Time-Series Simulation

7-day simulation at 1-hour resolution. Unlimited ECLSS shows baseline demands.

In [ ]:
hab_7d = Habitat(crew=crew, cabin_volume_m3=25.0, mission_duration_days=7)
ts = Simulation(habitat=hab_7d, system=EclssSystem.unlimited(), timestep_hours=1.0).run()

# Cabin atmosphere over time
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 6), sharex=True)
hours = ts.time_hours / 24  # convert to days

ax1.plot(hours, ts.cabin_o2_pp_kpa, color="#1565C0", linewidth=1.5)
ax1.axhspan(ymin=20.7, ymax=50.6, alpha=0.1, color="green", label="Safe range")
ax1.axhline(y=21.2, color="gray", linestyle="--", alpha=0.5, label="Target")
ax1.set_ylabel("O2 Partial Pressure (kPa)")
ax1.legend(loc="upper right")
ax1.set_title("Cabin Atmosphere Over 7 Days")

ax2.plot(hours, ts.cabin_co2_pp_kpa, color="#E65100", linewidth=1.5)
ax2.axhline(y=1.01, color="red", linestyle="--", alpha=0.5, label="SMAC limit")
ax2.axhline(y=0.507, color="gray", linestyle="--", alpha=0.5, label="Nominal")
ax2.set_ylabel("CO2 Partial Pressure (kPa)")
ax2.set_xlabel("Mission Day")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Per-crew O2 demand rates
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 4))
for name, arr in ts.crew_o2_demand_kg_hr.items():
    ax.plot(np.arange(len(arr)) / 24, arr, label=name, alpha=0.8)
ax.set_xlabel("Mission Day")
ax.set_ylabel("O2 Demand (kg/hr)")
ax.set_title("Per-Crew O2 Consumption Rate (Activity-Driven)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"O2 demand range: {ts.total_o2_demand_kg_hr[1:].min():.4f} - {ts.total_o2_demand_kg_hr[1:].max():.4f} kg/hr")
print(f"Peak-to-trough ratio: {ts.total_o2_demand_kg_hr[1:].max() / ts.total_o2_demand_kg_hr[1:].min():.1f}x")

## 4. Hardware Assessment: Undersized CO2 Scrubber

Set a CO2 removal rate that's too small and watch the buildup.

In [ ]:
# Half-capacity CO2 scrubber
undersized = EclssSystem(co2_removal_capacity_kg_hr=0.025)
ts_under = Simulation(habitat=hab_7d, system=undersized, timestep_hours=1.0).run()

# Compare CO2 buildup
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 4))
days = ts.time_hours / 24
ax.plot(days, ts.cabin_co2_pp_kpa, label="Unlimited ECLSS", color="#66BB6A")
ax.plot(ts_under.time_hours / 24, ts_under.cabin_co2_pp_kpa, label="Undersized scrubber", color="#EF5350")
ax.axhline(y=1.01, color="red", linestyle="--", alpha=0.5, label="SMAC limit (1.01 kPa)")
ax.axhline(y=0.507, color="orange", linestyle="--", alpha=0.3, label="Nominal (0.507 kPa)")
ax.set_xlabel("Mission Day")
ax.set_ylabel("CO2 Partial Pressure (kPa)")
ax.set_title("CO2 Buildup: Adequate vs Undersized CDRA")
ax.legend()
plt.tight_layout()
plt.show()

# Assessment
assessment = assess_system(ts=ts_under)
print(f"System verdict: {'PASS' if assessment.overall_pass else 'FAIL'}")
print(f"Violations: {len(assessment.violations)}")
if assessment.violations:
    v = assessment.violations[0]
    print(f"  {v.subsystem} exceeded from hour {v.start_hour:.0f} to {v.end_hour:.0f}")
    print(f"  Peak: {v.peak_value_kpa:.3f} kPa (limit: {v.limit_kpa:.3f})")

# Minimum requirements
reqs = MinimumRequirements.from_simulation(ts=ts)
print(f"\nMinimum hardware requirements (from unlimited baseline):")
print(f"  O2 generation:  {reqs.peak_o2_generation_kg_hr:.4f} kg/hr")
print(f"  CO2 removal:    {reqs.peak_co2_removal_kg_hr:.4f} kg/hr")
print(f"  Heat rejection: {reqs.peak_heat_rejection_w:.0f} W")

## 5. Trace Contaminant Assessment

17 BVAD contaminants checked against 180-day SMAC limits (Table 4-3).

In [ ]:
est = ContaminantEstimator.from_habitat(
    habitat=hab,
    equipment_mass_kg=2000.0,
)
smac = est.check_smac_limits(days=180)

# Plot: accumulation vs SMAC for the most critical contaminants
exceeded = [r for r in smac if r["exceeds_smac"]]
names_exc = [r["name"][:15] for r in exceeded]
accum = np.array([r["accumulation_mg_m3"] for r in exceeded])
limits = np.array([r["smac_mg_m3"] for r in exceeded])

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 5))
x = np.arange(len(names_exc))
ax.barh(x, accum, color="#EF5350", alpha=0.8, label="180-day accumulation")
ax.barh(x, limits, color="#66BB6A", alpha=0.4, label="SMAC limit")
ax.set_yticks(x)
ax.set_yticklabels(names_exc)
ax.set_xlabel("Concentration (mg/m3)")
ax.set_title(f"Contaminants Exceeding SMAC ({len(exceeded)} of 17)")
ax.legend()
ax.set_xscale("log")
plt.tight_layout()
plt.show()

print(f"\n{len(exceeded)} contaminants exceed SMAC without active TCCS removal.")
print("\nRequired removal rates for exceeded contaminants:")
for r in exceeded[:5]:
    print(f"  {r['name']:<25} {r['removal_rate_required_mg_day']:>8.1f} mg/day")

## 6. Schedule-Driven Simulation

ISS weekly pattern: 5 weekday + 2 weekend day rotation with different activity distributions.

In [ ]:
schedules = {cm.name: ISS_WEEKLY for cm in crew}
ts_iss = Simulation(
    habitat=hab_7d,
    schedules=schedules,
    system=EclssSystem.unlimited(),
    timestep_hours=1.0,
).run()

# Total heat over the week (shows daily exercise pattern)
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(12, 4))
days = ts_iss.time_hours[1:] / 24  # skip t=0
ax.plot(days, ts_iss.total_heat_w[1:], color="#FF6F00", linewidth=0.8)
ax.set_xlabel("Mission Day")
ax.set_ylabel("Total Heat Load (W)")
ax.set_title("Crew + Equipment Heat Load (ISS Weekly Schedule)")
# Mark weekends
for d in range(7):
    if d % 7 >= 5:
        ax.axvspan(xmin=d, xmax=d + 1, alpha=0.08, color="blue")
ax.legend(["Heat load", "Weekend"], loc="upper right")
plt.tight_layout()
plt.show()

# O2 demand variation
o2_min = ts_iss.total_o2_demand_kg_hr[1:].min()
o2_max = ts_iss.total_o2_demand_kg_hr[1:].max()
print(f"O2 demand: {o2_min:.4f} - {o2_max:.4f} kg/hr ({o2_max/o2_min:.1f}x variation)")